# Retrieval Bot: Move Selection via Game Database Retrieval

## Overview

The Retrieval Bot selects moves by searching a database of real human chess games for positions similar to the current one, then performing **weighted voting** across the moves those players chose.

Unlike the Human Bot, which generates moves dynamically using neural networks and a learned policy, the Retrieval Bot grounds its decisions in **observed human behavior**. Every move it plays was actually played by a real human in a real game.

The system has three main components:

* **Game database:** a collection of real games indexed by position
* **Retrieval hierarchy:** a three-level search that finds the most relevant games for the current position
* **Weighted voting:** a ranking function that scores candidate moves by how well the retrieved players match the target Elo and game outcome

**Inference pipeline:**

1. Encode the current board position as a vector
2. Search the database using the retrieval hierarchy
3. Score and rank candidate moves by weighted vote
4. Return the top-ranked legal move

---

## Core Principle

Human move prediction can be approached as a **nearest-neighbor problem** over a database of real games.

Given a query position and target Elo, we find games where:

* The position is similar (or identical) to the query
* The player's rating is close to the target Elo

We then ask: **what did those players do?**

The answer, weighted by relevance, becomes the bot's move.

This approach requires no training in the traditional sense. The "model" is the database itself, and the retrieval and weighting functions determine which evidence to trust.

---

## Data and Indexing

### Source Data

The database is built from `games.csv`, a CSV file of real games sourced from Lichess. Each row represents one complete game with fields for the move list, player ratings, whether the game was rated, and the result.

### Expanding Games into Positions

Each game is expanded into a sequence of per-position records using `_row_to_records()`. For every position reached during the game, the system stores:

* `full_fen` — the complete board state as a FEN string (including move counters)
* `partial_key` — a reduced key: piece placement, active turn, castling rights, en passant square (ignoring move counters)
* `board_vec` — a 785-dimensional float vector encoding the board (see Board Representation)
* `next_move_uci` — the move the player made from this position
* `player_elo` — the rating of the player whose turn it was
* `player_is_winner` — whether that player won the game
* `rated` — whether the game was rated

### Indices

At startup, three parallel data structures are built and cached to disk:

* **`full_fen_index`** — maps exact FEN strings to lists of matching records; supports Level 1 lookup
* **`partial_key_index`** — maps partial keys to lists of matching records; supports Level 2 lookup
* **`all_vectors`** — a NumPy array of all board vectors stacked row-wise; supports Level 3 cosine similarity search
* **`all_elos`** — a parallel array of player Elo ratings for fast Elo filtering

The cache is serialized to `retrieval_cache.pkl`. On subsequent runs, the cache is loaded directly rather than rebuilt from the CSV.

---

## Board Representation

Each board position is encoded as a **785-dimensional float vector** by `board_to_vector()`.

### Vector Layout

```
Dimensions 0 – 767    (12 × 64):   piece placement
Dimension  768         (1):          active turn
Dimensions 769 – 772   (4):          castling rights
Dimensions 773 – 784   (12):         piece counts
```

### Piece Placement (768 dims)

Twelve piece types are defined in a fixed order:

```
["P", "N", "B", "R", "Q", "K",   ← White pieces
 "p", "n", "b", "r", "q", "k"]   ← Black pieces
```

For each piece type (index `i`) and each square (index `s`), dimension `i * 64 + s` is set to 1.0 if that piece occupies that square, and 0.0 otherwise. This is a one-hot encoding across all 64 squares for each of the 12 piece types.

### Active Turn (1 dim)

Dimension 768 is 1.0 if it is White's turn, 0.0 if it is Black's turn.

### Castling Rights (4 dims)

Dimensions 769 through 772 encode the four castling rights as binary flags:

```
769: White kingside
770: White queenside
771: Black kingside
772: Black queenside
```

### Piece Counts (12 dims)

Dimensions 773 through 784 store the total count of each piece type on the board. This is redundant with the placement encoding but provides a compact signal that is useful for similarity search in the Level 3 fallback.

---

## Retrieval Hierarchy

Move recommendation is handled by `recommend_move()`, which searches in three levels and returns at the first level that finds enough candidates.

### Level 1 — Exact FEN Match

```python
exact_candidates = full_fen_index.get(board.fen(), [])
```

Looks up the complete FEN string, including move counters. Filters by Elo window of ±100. If fewer than 5 candidates are found, the window relaxes to ±200.

If at least 5 candidates remain, weighted voting is performed and results are returned.

**When it fires:** the current position has been seen before in the database, including the same move number and repetition context.

---

### Level 2 — Partial Key Match

```python
partial_key = fen_to_partial_key(board)
partial_candidates = partial_key_index.get(partial_key, [])
```

The partial key captures piece placement, active turn, castling rights, and en passant square, but ignores the half-move clock and full-move counter. This means the same board configuration reached via different move orders matches here even if it did not match at Level 1.

Elo filtering is identical to Level 1: ±100, relaxing to ±200.

**When it fires:** the position is a transposition or occurred at a different move number than stored in the database.

---

### Level 3 — Cosine Similarity Search

```python
sim_candidates, sims = retrieve_similar_records(board, query_elo, top_n=300, elo_window=200)
```

When no exact or partial key match is found, the system computes cosine similarity between the query board vector and all stored board vectors within an Elo window of ±200. The top 300 most similar positions are retrieved, along with their similarity scores. These scores are passed to the voting function as `w_similarity`.

**When it fires:** the position is novel or unusual and does not appear in the database under any transposition.

---

## Weighting and Ranking

All retrieved records are scored by `record_weight()` and aggregated by `aggregate_move_scores()`.

### Weight Formula

```
weight = w_elo * w_similarity * w_win * w_rated
```

Each factor captures a different dimension of relevance:

* **`w_elo`** — exponential decay by Elo distance:

```python
w_elo = exp(-|player_elo - query_elo| / 150)
```

  A player 150 Elo away contributes roughly 37% of the weight of an exact match. A player 300 Elo away contributes about 14%.

* **`w_similarity`** — cosine similarity between the stored board vector and the query vector. Set to 1.0 for Levels 1 and 2, where the position is already an exact or near-exact match.

* **`w_win`** — 1.15 if the player won the game, 1.0 otherwise. Winning players are given slightly more trust on the assumption that they were playing better chess in that game than their rating alone suggests.

* **`w_rated`** — 1.05 if the game was rated, 1.0 otherwise. Rated games between matched players reflect more deliberate play.

### Vote Aggregation

Weights are summed across all records for each candidate move. Only moves that are legal in the current position receive weight. Scores are normalized to a probability distribution over the legal move set.

The top-ranked move is returned as the bot's choice. The top three moves and their probabilities are also exposed via `get_last_top_choices()` for display in the frontend.

---

## Inference Pipeline

Given a position and target Elo:

1. **Load cache** (once at startup):

```python
load_or_build_cache()
```

2. **Retrieve candidates** using the three-level hierarchy

3. **Aggregate weighted votes** across all retrieved records

4. **Apply think delay:**

```python
think_time = self.compute_think_time(board, whiteMs, blackMs)
time.sleep(think_time)
```

   Delay = clamp(remaining_seconds × 0.03, 0.05s, 2.0s)

5. **Return top-1 move:**

```python
return chess.Move.from_uci(results[0]["uci"])
```

If retrieval returns no candidates at all (the database is empty or the cache is not loaded), the bot falls back to the first legal move available.

---

## Design Rationale

### 1. Three-Level Hierarchy

A single lookup strategy is not robust across all positions. Exact FEN lookup is highly precise but misses transpositions. Cosine similarity alone is noisy for common positions. The three-level design uses the strongest signal available:

* Exact transposition data when it exists
* Structural equivalence (same configuration, different move order) as a first fallback
* Approximate similarity as a last resort

Each level is tried in decreasing precision order.

---

### 2. Elo Filtering Before Similarity Search

At Level 3, the Elo filter is applied **before** cosine similarity is computed. This is intentional. Retrieving the most similar positions from the entire database and then filtering by Elo would waste compute on irrelevant players. Filtering first reduces the candidate pool to players at a comparable level before similarity is evaluated.

---

### 3. Exponential Elo Decay (tau = 150)

The Elo weight uses an exponential decay with a time constant of 150 rating points. This reflects that chess playing strength scales with Elo difference: a 150-point advantage corresponds to approximately a 70% expected win rate. The decay rate is matched to the natural scale of meaningful Elo differences.

---

### 4. Winner Bonus

Winning players are weighted at 1.15 times the base weight. Players who won the game were likely playing better chess in that game than their rating alone suggests, so their move choices carry slightly more predictive value.

---

### 5. No Sampling

The Human Bot samples from a probability distribution over engines, introducing move-to-move variability. The Retrieval Bot always plays the top-weighted move. This is consistent with its purpose: it is a **retrieval system**, not a generative one. Its predictions reflect what the database says is most commonly played at the target level from similar positions.

---

## Resulting Behavior

The Retrieval Bot:

* Plays moves drawn from **real human games** at the target Elo
* Handles **common positions** with high fidelity via exact and transposition matches
* Degrades gracefully to **approximate similarity** for unusual or novel positions
* Produces **consistent, deterministic** move choices given the same position and Elo
* Surfaces its top three candidate moves and probabilities for display in the frontend

Its behavior is best understood as the answer to: **"what do real players at this level typically play here?"**

---

## Summary

The Retrieval Bot models chess play as:

> A nearest-neighbor retrieval over a database of real human games, with candidate moves ranked by weighted vote based on Elo proximity, position similarity, game outcome, and whether the game was rated.

It achieves human-like play by:

* grounding every move in **observed human decisions**
* weighting evidence by **how relevant each retrieved game is**
* using a **hierarchical retrieval strategy** that degrades gracefully from exact to approximate matching